# Filter Evals   
---   
This code looks in the `model_outputs` folder and for each folder, it runs through the various files and creates two outputs:
1) Correct Solutions
2) Incorrect Solutions

In [1]:
import json
from pathlib import Path
from utils.parsing import extract_solution, coerce_response
from utils.exceptions import IllegalMoveException

In [2]:
# Toggle whether to write out JSONL files
SAVE_FILES = False

def evaluate_file(fp: Path):
    task = fp.stem.split('_')[0]
    res = {"Correct": 0, "Incorrect": 0, "Thrown Away": 0}
    correct, incorrect, errors = [], [], []
    data = json.load(fp.open())
    for e in data:
        try:
            prompt = e["prompt"]
            resp   = e["model_response"]
            gt     = e["info"]["answer"]

            if task in ("bestmove", "worstmove"):
                answer     = gt["answer"]
                candidates = gt["candidates"]
                pred = coerce_response(extract_solution(resp), "choose_from_n")
                if pred == answer:
                    res["Correct"] += 1
                    correct.append(e)
                elif pred in candidates:
                    res["Incorrect"] += 1
                    incorrect.append(e)
                else:
                    raise IllegalMoveException()

            elif task == "legalmoves":
                pred = coerce_response(extract_solution(resp), "produce_list")
                if set(pred) == set(gt) and len(pred) == len(gt):
                    res["Correct"] += 1
                    correct.append(e)
                else:
                    res["Incorrect"] += 1
                    incorrect.append(e)

            elif task == "predictmove":
                pred = coerce_response(extract_solution(resp), "predict_singlemove")
                if pred in gt:
                    sorted_gt = sorted(gt.items(), key=lambda x: x[1])
                    idx = next(i for i, (m, _) in enumerate(sorted_gt) if m == pred)
                    rank = idx / len(sorted_gt)
                    if rank > 0.7:
                        res["Correct"] += 1
                        correct.append(e)
                    else:
                        res["Incorrect"] += 1
                        incorrect.append(e)
                else:
                    raise IllegalMoveException()

            else:
                # Skip unknown tasks
                continue

        except Exception:
            res["Thrown Away"] += 1
            errors.append(e)

    return task, res, correct, incorrect, errors

base = Path("model_outputs")
for folder in base.iterdir():
    if not folder.is_dir():
        continue

    agg = {"Correct": 0, "Incorrect": 0, "Thrown Away": 0}
    all_corr, all_incorr, all_errs = [], [], []

    for fp in folder.glob("*.json"):
        task, res, corr, incorr, errs = evaluate_file(fp)
        agg["Correct"]     += res["Correct"]
        agg["Incorrect"]   += res["Incorrect"]
        agg["Thrown Away"] += res["Thrown Away"]

        all_corr.extend(corr)
        all_incorr.extend(incorr)
        all_errs.extend(errs)

        if SAVE_FILES:
            # write per-file JSONL
            with (folder / f"{fp.stem}_correct.jsonl").open("w") as f:
                for e in corr:
                    f.write(json.dumps(e) + "\n")
            with (folder / f"{fp.stem}_incorrect.jsonl").open("w") as f:
                for e in incorr:
                    f.write(json.dumps(e) + "\n")
            with (folder / f"{fp.stem}_errors.jsonl").open("w") as f:
                for e in errs:
                    f.write(json.dumps(e) + "\n")

    def avg_len(lst):
        return sum(len(e["model_response"]) for e in lst) / len(lst) if lst else 0

    print(f"{folder.name}: {agg['Correct']} correct, {agg['Incorrect']} incorrect, {agg['Thrown Away']} thrown away")
    print(f"  Avg model_response length (chars): correct={avg_len(all_corr):.1f}, "
          f"incorrect={avg_len(all_incorr):.1f}, errors={avg_len(all_errs):.1f}")

llmchess-llama31-8b-400: 203 correct, 878 incorrect, 519 thrown away
  Avg model_response length (chars): correct=1472.7, incorrect=1481.8, errors=1068.2
llmchess-llama31-8b-sft-mmxl-400: 440 correct, 893 incorrect, 267 thrown away
  Avg model_response length (chars): correct=3068.5, incorrect=3441.3, errors=3451.4
llmchess-llama4-maverick-400: 340 correct, 925 incorrect, 335 thrown away
  Avg model_response length (chars): correct=2930.1, incorrect=3001.3, errors=2755.3
llmchess-qwen25-7b-400: 175 correct, 861 incorrect, 564 thrown away
  Avg model_response length (chars): correct=1150.3, incorrect=1085.9, errors=853.5
llmchess-qwen25-7b-grpo-datamix-2-400: 563 correct, 886 incorrect, 151 thrown away
  Avg model_response length (chars): correct=1531.0, incorrect=1870.2, errors=3362.3
llmchess-qwen25-7b-grpo-mmxl-400: 410 correct, 1018 incorrect, 172 thrown away
  Avg model_response length (chars): correct=3305.9, incorrect=3552.8, errors=5509.1
llmchess-qwen25-7b-sft-dm2-v2-400: 373 c